In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "5"

import sys
from pathlib import Path

import equinox as eqx
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

sys.path.insert(0, str(Path(".").resolve()))

from data_loader import load_fire_event
from model import WildfireModel
from simulation import run_simulation_no_grad
from training import make_initial_state
from utils import load_config

print("JAX devices:", jax.devices())

PLOTS_DIR = Path("../result_saved/paper_plots")
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Plots dir: {PLOTS_DIR.resolve()}")


In [ ]:
events_model = {
    "Buck_2017": "result_saved/best_model",
    "Pier_2017": "result_saved/best_model",
    "Bear_2020": "result_saved/best_model",
    "Chimney_2016": "result_saved/best_model",
    "Ferguson_2018": "result_saved/best_model",
    "Brattain_2020": "result_saved/best_model",
}
Pth = 0.5
days = [0, 20]
x_train_last_day = 10
model_file_name = "best_loss.eqx"  # best_loss.eqx, best_iou.eqx

print(f"Pth: {Pth}")
print(f"Model: {model_file_name}")
print(f"Events: {list(events_model.keys())}")

In [ ]:
def compute_event_metrics(model, config, event_name, event_info, Pth):
    """Run simulation for one event, return per-day metrics and raw arrays."""
    event = load_fire_event(event_name, config, event_info)
    fire_seq = event.fire_seq
    initial_fire = jnp.array(fire_seq[0])
    p_u, p_b, p_bd = make_initial_state(initial_fire)

    daily_pfire = run_simulation_no_grad(
        model,
        p_u,
        p_b,
        p_bd,
        jnp.array(event.static_cnn_input),
        jnp.array(event.slope_ca),
        jnp.array(event.aspect_upslope),
        jnp.array(event.fuel_type_map),
        jnp.array(event.wind_u),
        jnp.array(event.wind_v),
        jnp.array(event.fuel_mask),
        config,
    )

    fire_pred = np.clip(np.array(daily_pfire), 0.0, 1.0)
    P_pred = np.concatenate([np.array(fire_seq[0])[None, ...], fire_pred], axis=0)
    target = np.array(fire_seq)
    initial_mask = (target[0] > Pth).astype(float)

    day_start = event_info["days"][0]
    day0 = day_start - 1 if day_start > 0 else 0
    actual_days = [day0 + i for i in range(P_pred.shape[0])]

    pred_px, target_px, manhattan, ious = [], [], [], []
    for i in range(P_pred.shape[0]):
        pred_new = (P_pred[i] > Pth).astype(float) * (1 - initial_mask)
        tgt_new = (target[i] > Pth).astype(float) * (1 - initial_mask)
        pf, tf = int(pred_new.sum()), int(tgt_new.sum())
        tp = ((pred_new == 1) & (tgt_new == 1)).sum()
        fp = ((pred_new == 1) & (tgt_new == 0)).sum()
        fn = ((pred_new == 0) & (tgt_new == 1)).sum()
        iou = float(tp / (tp + fp + fn + 1e-7))
        pred_px.append(pf)
        target_px.append(tf)
        manhattan.append(abs(pf - tf))
        ious.append(iou)

    return {
        "actual_days": actual_days,
        "pred_px": pred_px,
        "target_px": target_px,
        "manhattan": manhattan,
        "ious": ious,
        "P_pred": P_pred,
        "target": target,
    }


def plot_event_days(m, event_name, Pth, days_per_row=7):
    """Plot Prediction, Target, and Diff (red=FP, green=FN) per day.

    Layout per figure: 3 rows × up to ``days_per_row`` columns
    (row 1 = Prediction, row 2 = Target, row 3 = Diff).
    Days are split into chunks so each figure stays legible.
    """
    P_pred = m["P_pred"][1:]
    target = m["target"][1:]
    actual_days = m["actual_days"][1:]
    n_frames = len(P_pred)
    if n_frames == 0:
        return

    n_chunks = (n_frames + days_per_row - 1) // days_per_row

    title_fs = 32
    label_fs = 32

    for chunk in range(n_chunks):
        start = chunk * days_per_row
        end = min(start + days_per_row, n_frames)
        n_cols = end - start

        # Size the figure so each axes slot matches the image aspect ratio
        # exactly. Without this, matplotlib's aspect="equal" letterboxes the
        # image inside the slot, leaving visible whitespace between rows even
        # with hspace=0.
        img_h, img_w = P_pred[start].shape
        panel_w = 4.2
        panel_h = panel_w * img_h / img_w
        left_in, right_in = 1.0, 0.05
        top_in, bottom_in = 0.6, 0.05
        fig_w = panel_w * n_cols + left_in + right_in
        fig_h = panel_h * 3 + top_in + bottom_in

        fig, axes = plt.subplots(3, n_cols, figsize=(fig_w, fig_h), squeeze=False)

        for col in range(n_cols):
            f = start + col
            ad = actual_days[f]
            pred_binary = (P_pred[f] > Pth).astype(float)
            target_binary = (target[f] > Pth).astype(float)
            diff = np.zeros((*pred_binary.shape, 3))
            diff[:, :, 0] = np.maximum(pred_binary - target_binary, 0)
            diff[:, :, 1] = np.maximum(target_binary - pred_binary, 0)

            ax_pred = axes[0, col]
            ax_pred.imshow(pred_binary, vmin=0, vmax=1, cmap="Reds", origin="upper")
            ax_pred.set_title(f"Day {ad}", fontsize=title_fs)
            ax_pred.set_xticks([])
            ax_pred.set_yticks([])

            ax_tgt = axes[1, col]
            ax_tgt.imshow(target[f], vmin=0, vmax=1, cmap="Reds", origin="upper")
            ax_tgt.set_xticks([])
            ax_tgt.set_yticks([])

            ax_diff = axes[2, col]
            ax_diff.imshow(diff, origin="upper")
            ax_diff.set_xticks([])
            ax_diff.set_yticks([])

        axes[0, 0].set_ylabel("Prediction", fontsize=label_fs, fontweight="bold")
        axes[1, 0].set_ylabel("Target", fontsize=label_fs, fontweight="bold")
        axes[2, 0].set_ylabel("red=FP\ngreen=FN", fontsize=label_fs, fontweight="bold")

        # Lock axes slots to absolute inch margins so each slot matches the
        # image aspect ratio exactly — zero gap between rows.
        fig.subplots_adjust(
            left=left_in / fig_w,
            right=1.0 - right_in / fig_w,
            bottom=bottom_in / fig_h,
            top=1.0 - top_in / fig_h,
            wspace=0.02,
            hspace=0.0,
        )
        fname = PLOTS_DIR / f"{event_name}_{chunk + 1}.png"
        fig.savefig(fname, dpi=150, bbox_inches="tight")
        print(f"  saved {fname.name}")
        plt.show()


all_metrics = {}
for ev, ev_dir in events_model.items():
    in_dir = Path("..") / ev_dir
    config_file = in_dir / "config.yaml"
    model_path = in_dir / model_file_name

    if not config_file.exists():
        print(f"Skipping {ev}: config not found at {config_file}")
        continue
    if not model_path.exists():
        print(f"Skipping {ev}: model not found at {model_path}")
        continue

    config = load_config(str(config_file))
    config["data"]["base_dir"] = "../" + config["data"]["base_dir"]

    all_events_cfg = {}
    all_events_cfg.update(config["data"].get("train_events") or {})
    all_events_cfg.update(config["data"].get("test_events") or {})

    if ev not in all_events_cfg:
        print(f"Skipping {ev}: not in config at {config_file}")
        continue

    ev_info = dict(all_events_cfg[ev])
    ev_info["days"] = days

    key = jax.random.PRNGKey(config["model"]["seed"])
    model = WildfireModel(config, key)
    model = eqx.tree_deserialise_leaves(str(model_path), model)
    model = eqx.tree_inference(model, value=True)

    print(f"Running {ev} ({ev_dir})...", end=" ")
    all_metrics[ev] = compute_event_metrics(model, config, ev, ev_info, Pth)
    n = len(all_metrics[ev]["ious"]) - 1
    avg_iou = np.mean(all_metrics[ev]["ious"][1:])
    print(f"{n} days, avg IoU={avg_iou:.4f}")

## 2×2 Metric Panels — IoU · Manhattan · Precision · Recall

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9), dpi=150)
ax_iou, ax_man = axes[0, 0], axes[0, 1]
ax_prec, ax_rec = axes[1, 0], axes[1, 1]

# Match colors with the existing 1d Manhattan/IoU plot.
colors = plt.cm.tab10(np.linspace(0, 1, len(all_metrics)))

all_pred_days = set()
for ev, m in all_metrics.items():
    all_pred_days.update(m["actual_days"][1:])
all_pred_days = sorted(all_pred_days)

for (ev, m), color in zip(all_metrics.items(), colors):
    pred_days = m["actual_days"][1:]
    ious_s = m["ious"][1:]
    manhattan_s = [v / 1e4 for v in m["manhattan"][1:]]

    # Recompute precision/recall per day using the SAME masking convention
    # (exclude initial-mask pixels, threshold at Pth) as compute_event_metrics
    # so IoU/Manhattan panels stay identical to the existing 1d figure.
    P_pred = m["P_pred"]
    target = m["target"]
    initial_mask = (target[0] > Pth).astype(float)

    precisions, recalls = [], []
    for i in range(1, len(P_pred)):
        pred_new = (P_pred[i] > Pth).astype(float) * (1 - initial_mask)
        tgt_new = (target[i] > Pth).astype(float) * (1 - initial_mask)
        pf, tf = int(pred_new.sum()), int(tgt_new.sum())
        if tf == 0 and pf == 0:
            # Both masks empty — undefined, skip the point.
            precisions.append(np.nan)
            recalls.append(np.nan)
            continue
        tp = int(((pred_new == 1) & (tgt_new == 1)).sum())
        fp = int(((pred_new == 1) & (tgt_new == 0)).sum())
        fn = int(((pred_new == 0) & (tgt_new == 1)).sum())
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        precisions.append(prec)
        recalls.append(rec)

    plot_kw = dict(color=color, label=ev, markersize=4, linewidth=1.5)
    ax_iou.plot(pred_days, ious_s, "o-", **plot_kw)
    ax_man.plot(pred_days, manhattan_s, "o-", **plot_kw)
    ax_prec.plot(pred_days, precisions, "o-", **plot_kw)
    ax_rec.plot(pred_days, recalls, "o-", **plot_kw)

panels = [
    (ax_iou, "(a) IoU", "IoU", (0.0, 1.0)),
    (ax_man, "(b) Manhattan Distance", r"Manhattan Distance ($\times 10^4$)", None),
    (ax_prec, "(c) Precision", "Precision", (0.0, 1.0)),
    (ax_rec, "(d) Recall", "Recall", (0.0, 1.0)),
]
for ax, title, ylabel, ylim in panels:
    ax.axvline(
        x=x_train_last_day,
        color="red",
        linestyle="--",
        linewidth=1.5,
        alpha=0.6,
        label=f"Train last day ({x_train_last_day})",
    )
    ax.set_xlabel("Day", fontsize=13)
    ax.set_xticks(all_pred_days)
    ax.grid(True, alpha=0.3)
    ax.set_ylabel(ylabel, fontsize=14)
    ax.set_title(title, fontsize=14, fontweight="bold")
    if ylim is not None:
        ax.set_ylim(*ylim)

fig.suptitle(f"All Events (Pth={Pth})", fontsize=16, fontweight="bold")

# Reserve room at the bottom for the shared legend, then anchor the legend
# box to the same horizontal span as the plots so its width matches.
left_margin, right_margin = 0.07, 0.98
fig.subplots_adjust(
    left=left_margin,
    right=right_margin,
    top=0.94,
    bottom=0.12,
    wspace=0.20,
    hspace=0.28,
)
handles, labels = ax_iou.get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="lower left",
    bbox_to_anchor=(left_margin, 0.01, right_margin - left_margin, 0.075),
    mode="expand",
    ncol=len(labels),
    fontsize=12,
    frameon=True,
)

fname = PLOTS_DIR / "all_events_metrics.png"
fig.savefig(fname, dpi=150, bbox_inches="tight")
print(f"saved {fname.name}")
plt.show()

## Summary Table

In [ ]:
print(f"{'Event':<20} {'Avg IoU':>8} {'Avg Manhattan':>14} {'Total Manhattan':>16}")
print("-" * 62)
for ev, m in all_metrics.items():
    ious_s = m["ious"][1:]
    manhattan_s = m["manhattan"][1:]
    print(f"{ev:<20} {np.mean(ious_s):>8.4f} {np.mean(manhattan_s):>14.0f} {sum(manhattan_s):>16,}")